# Attention Profiling on GPU

## 1. Verify/Install Correct NSight Version

In [1]:
!nvcc --version

/bin/bash: line 1: nvcc: command not found


In [2]:
!ncu --version

/bin/bash: line 1: ncu: command not found


For RTX 2060 GPU:
* CUDA V12.6.85
* NSight 2024.3.2.0

For T4 GPU:
* CUDA V12.8.93
* NSight 2025.1.1

## 2. Testing Script

Make sure it works before running it all at once with NSight

### Setup

In [3]:
import numpy as np

import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

In [4]:
# following FlashAttention-3 paper
def generate_matrix(shape, seed=None) -> np.ndarray:
    if seed is not None:
        np.random.seed(seed)
    # Base matrix from N(0, 1)
    base = np.random.normal(loc=0.0, scale=1.0, size=shape)
    # Bernoulli mask (0.001 probability of being 1)
    mask = np.random.binomial(n=1, p=0.001, size=shape)
    # Noise from N(0, 100)
    noise = np.random.normal(loc=0.0, scale=10.0, size=shape)
    # Final matrix: base + noise * mask
    return base + noise * mask

In [5]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
device

device(type='cuda')

In [6]:
def scaled_dot_product_attention(Q_np: np.ndarray, K_np: np.ndarray, V_np: np.ndarray, causal: bool) -> np.ndarray:
    # ensure matching dimensions of 4D tensors
    assert (len(Q_np.shape), len(K_np.shape), len(V_np.shape)) == (4, 4, 4)
    b, h, seq_q, d = Q_np.shape
    bk, hk, seq_k, dk = K_np.shape
    bv, hv, seq_v, dv = V_np.shape
    assert b == 1 and b == bk and b == bv
    assert h == 1 and h == hk and h == hv
    assert d == dk, "Q and K head dim must be equal"
    assert d == dv, f"Q ({d}) and V ({dv}) head dim must be equal"
    assert seq_k == seq_v, "K and V must have equal seq len"

    # use CUDA on GPU
    device = torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
    )

    Q_torch = torch.from_numpy(Q_np).to(device)
    K_torch = torch.from_numpy(K_np).to(device)
    V_torch = torch.from_numpy(V_np).to(device)

    # for Turing arch, cannot use FlashAttention2
    with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
        O_torch = F.scaled_dot_product_attention(Q_torch, K_torch, V_torch,
                                                 attn_mask=None,  # no masking
                                                 dropout_p=0.0,  # no dropout
                                                 is_causal=causal)

    ##### Use torch profiler to see which CUDA kernel it is using for attention
    # with torch.profiler.profile(
    #     activities=[torch.profiler.ProfilerActivity.CUDA],
    #     record_shapes=True,
    #     with_stack=False
    # ) as prof:
    #     O_torch = F.scaled_dot_product_attention(Q_torch, K_torch, V_torch, attn_mask=None, dropout_p=0.0, is_causal=causal)
    #     torch.cuda.synchronize()
    
    # print(prof.key_averages().table(sort_by="cuda_time_total", max_name_column_width=200))
    # torch.cuda.empty_cache()
    # torch.cuda.reset_peak_memory_stats()
    # torch.cuda.synchronize()
    #####

    return O_torch.cpu().numpy()

In [7]:
seq_q = seq_kv = 256
d = 64
seed = 42
causal = False

### Create numpy arrays

Must be `batch_size x num_heads x seq_len x head_dim` for memory-efficient attention.

If it is 2D (`seq_len x head_dim` only), torch will fall back to the Math implementation that has no fused operations.

In [8]:
# Use FP16 for FA1 or MemEff Attention
# Ensure 4D with correct axes to match MemEff implementation
Q_np = generate_matrix((seq_q, d), seed=seed).astype(np.float16)[np.newaxis, np.newaxis, :, :]
K_np = generate_matrix((seq_kv, d), seed=seed).astype(np.float16)[np.newaxis, np.newaxis, :, :]
V_np = generate_matrix((seq_kv, d), seed=seed).astype(np.float16)[np.newaxis, np.newaxis, :, :]
Q_np.shape, K_np.shape, V_np.shape

((1, 1, 256, 64), (1, 1, 256, 64), (1, 1, 256, 64))

RTX 2060 and T4 (Turing) use CUDA Kernel:

```
fmha_cutlassF_f16_aligned_64x64_rf_sm75(PyTorchMemEffAttention::AttentionKernel<cutlass::half_t, cutlass::arch::Sm75, true, 64, 64, 64, true, true>::Params)
```

### Run the attention algorithm

In [9]:
scaled_dot_product_attention(Q_np, K_np, V_np, causal)

array([[[[ 0.3523 , -0.154  ,  0.4583 , ..., -0.1431 , -0.835  ,
          -0.8315 ],
         [ 0.709  ,  1.149  , -0.07654, ...,  1.875  , -0.9097 ,
          -0.4468 ],
         [ 0.076  , -0.4434 , -1.147  , ..., -1.183  , -0.3625 ,
           0.6724 ],
         ...,
         [-1.357  , -0.3696 ,  1.829  , ...,  0.761  , -0.9473 ,
          -0.2542 ],
         [-1.069  , -2.39   ,  0.957  , ..., -0.1417 , -0.8896 ,
           1.398  ],
         [ 0.79   , -1.08   ,  0.02559, ...,  0.2837 , -0.8184 ,
           0.2563 ]]]], shape=(1, 1, 256, 64), dtype=float16)

FlashAttention currently supports:

Turing, Ampere, Ada, or Hopper GPUs (e.g., H100, A100, RTX 3090, T4, RTX 2080).
fp16 and bf16 (bf16 requires Ampere, Ada, or Hopper GPUs).
Head dimensions that are multiples of 8, up to 128 (e.g., 8, 16, 24, ..., 128). Head dim > 64 backward requires A100 or H100.

## 3. Run Testing Script with NSight

### Find the correct kernel and Number of Kernels to skip

In [10]:
!ncu --print-summary per-kernel ~/thesis-gpu/.venv/bin/python testing_script.py

/bin/bash: line 1: ncu: command not found


### Profile the Kernel

#### Gets a lot of metrics that are irrelevant, just make sure it's hitting the correct kernel

In [11]:
!ncu --set full --launch-skip 5 --launch-count 1 -o profile_test_full ~/thesis-gpu/.venv/bin/python testing_script.py

/bin/bash: line 1: ncu: command not found


#### Get the correct metrics to profile

All metrics can be found in the [Metrics Reference documentation](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-reference)

In [12]:
s = """
sm__pipe_tensor_cycles_active.avg,
smsp__inst_executed_pipe_tensor.sum,
sm__pipe_fma_cycles_active.avg,
smsp__inst_executed_pipe_fma.sum,
smsp__inst_executed_pipe_xu.sum,
smsp__inst_executed_pipe_xu.avg.pct_of_peak_sustained_elapsed,
sm__pipe_alu_cycles_active.avg,
smsp__inst_executed_pipe_alu.sum,
sm__pipe_lsu_cycles_active.avg,
smsp__inst_executed_pipe_lsu.sum,
dram__cycles_active.avg,
lts__cycles_active.avg,
sm__inst_executed.sum,
sm__cycles_elapsed.avg,
gpu__time_duration.sum,
smsp__warps_issue_stalled_short_scoreboard.avg,
smsp__warps_issue_stalled_long_scoreboard.avg,
smsp__warps_issue_stalled_wait.avg,
smsp__warps_issue_stalled_math_pipe_throttle.avg,
smsp__warps_issue_stalled_mio_throttle.avg,
smsp__warps_issue_stalled_not_selected.avg
"""

metrics_to_measure = s.replace("\n", "").split(",")
metrics_to_measure = [m.strip() for m in metrics_to_measure]
print(','.join(metrics_to_measure))


sm__pipe_tensor_cycles_active.avg,smsp__inst_executed_pipe_tensor.sum,sm__pipe_fma_cycles_active.avg,smsp__inst_executed_pipe_fma.sum,smsp__inst_executed_pipe_xu.sum,smsp__inst_executed_pipe_xu.avg.pct_of_peak_sustained_elapsed,sm__pipe_alu_cycles_active.avg,smsp__inst_executed_pipe_alu.sum,sm__pipe_lsu_cycles_active.avg,smsp__inst_executed_pipe_lsu.sum,dram__cycles_active.avg,lts__cycles_active.avg,sm__inst_executed.sum,sm__cycles_elapsed.avg,gpu__time_duration.sum,smsp__warps_issue_stalled_short_scoreboard.avg,smsp__warps_issue_stalled_long_scoreboard.avg,smsp__warps_issue_stalled_wait.avg,smsp__warps_issue_stalled_math_pipe_throttle.avg,smsp__warps_issue_stalled_mio_throttle.avg,smsp__warps_issue_stalled_not_selected.avg


#### Construct commands with correct metrics and parameters

In [13]:
_raw = torch.cuda.get_device_name(0).lower()
for _prefix in ["nvidia geforce ", "nvidia ", "geforce ", "tesla ", "quadro "]:
    if _raw.startswith(_prefix):
        _raw = _raw[len(_prefix):]
        break
gpu_name = _raw.replace(" ", "")
print(f"GPU: {torch.cuda.get_device_name(0)} -> slug: {gpu_name}")

GPU: NVIDIA GeForce RTX 2060 -> slug: rtx2060


In [14]:
seq_lens = [256, 512, 1024, 2048]
head_dims = [64]

In [15]:
cmds = []

for seq_len in seq_lens:
    for head_dim in head_dims:
        metrics_str = ','.join(metrics_to_measure)
        output_name = f"profile_{gpu_name}_{seq_len}x{head_dim}"
        cmd = (
            f"ncu --metrics {metrics_str}"
            f" --launch-skip 5 --launch-count 1"
            f" -o {output_name}"
            f" ~/thesis-gpu/.venv/bin/python testing_script.py"
            f" --seq_q {seq_len} --seq_kv {seq_len} --d {head_dim}"
        )
        cmds.append(cmd)
        print(f'{gpu_name} {seq_len}x{head_dim}:')
        print(cmd)
        print()

rtx2060 256x64:
ncu --metrics sm__pipe_tensor_cycles_active.avg,smsp__inst_executed_pipe_tensor.sum,sm__pipe_fma_cycles_active.avg,smsp__inst_executed_pipe_fma.sum,smsp__inst_executed_pipe_xu.sum,smsp__inst_executed_pipe_xu.avg.pct_of_peak_sustained_elapsed,sm__pipe_alu_cycles_active.avg,smsp__inst_executed_pipe_alu.sum,sm__pipe_lsu_cycles_active.avg,smsp__inst_executed_pipe_lsu.sum,dram__cycles_active.avg,lts__cycles_active.avg,sm__inst_executed.sum,sm__cycles_elapsed.avg,gpu__time_duration.sum,smsp__warps_issue_stalled_short_scoreboard.avg,smsp__warps_issue_stalled_long_scoreboard.avg,smsp__warps_issue_stalled_wait.avg,smsp__warps_issue_stalled_math_pipe_throttle.avg,smsp__warps_issue_stalled_mio_throttle.avg,smsp__warps_issue_stalled_not_selected.avg --launch-skip 5 --launch-count 1 -o profile_rtx2060_256x64 ~/thesis-gpu/.venv/bin/python testing_script.py --seq_q 256 --seq_kv 256 --d 64

rtx2060 512x64:
ncu --metrics sm__pipe_tensor_cycles_active.avg,smsp__inst_executed_pipe_tensor.

#### Run the profiler with the correct metrics

In [16]:
for cmd in cmds:
    !{cmd}

/bin/bash: line 1: ncu: command not found
/bin/bash: line 1: ncu: command not found
/bin/bash: line 1: ncu: command not found
/bin/bash: line 1: ncu: command not found


### View Profile Results

Instructions and example code for profiler API found in the [Python Report Interface documentation](https://docs.nvidia.com/nsight-compute/PythonReportInterface/index.html)

#### Setup package

In [17]:
!python -m pip install jupyterlab-nvidia-nsight

Add the directory for the `ncu_report` package in NSight to the PYTHON PATH

In [18]:
import subprocess
result = subprocess.run(['find', '/usr', '/opt', '-name', 'ncu_report*',
'-type', 'f'],
                        capture_output=True, text=True)

path = result.stdout.splitlines()[0][:-len('ncu_report.py')]
print(path)

import sys                                                                  
sys.path.append(path)
import ncu_report

/opt/nvidia/nsight-compute/2024.3.2/extras/python/


#### Get the report info

In [19]:
REPORT_NAME = "profile_rtx2060_1024x64.ncu-rep"

my_context = ncu_report.load_report(REPORT_NAME)
my_context.num_ranges()

1

In [20]:
my_range = my_context.range_by_idx(0)
my_range.num_actions()

1

In [21]:
my_action = my_range.action_by_idx(0)
my_action.name()

'fmha_cutlassF_f16_aligned_64x64_rf_sm75'

In [22]:
# Get action metric names : values
my_metric_names = my_action.metric_names()

my_metric_values_string = list(
    filter(lambda x: x is not None or True,
    map(lambda name: my_action.metric_by_name(name).as_string(), my_metric_names)
    ))
my_metric_values_float = list(
    filter(lambda x: x != 0.0 or True,
    map(lambda name: my_action.metric_by_name(name).as_double(), my_metric_names)
    ))

# replace any string values that are None with the floats, leave the rest of the strings as is
my_metric_values = [float(my_metric_values_float[i]) if (val is None or float(my_metric_values_float[i]) != 0.0) else val for i, val in enumerate(my_metric_values_string)]


my_metrics = dict(zip(my_metric_names, my_metric_values))
my_metrics_pruned = dict([(key, my_metrics[key]) for key in my_metrics.keys() if key in metrics_to_measure])

print(f"Number of missing metrics: {len(metrics_to_measure) - len(my_metrics_pruned)}")
print(f"Missing metrics:")
for i, key in enumerate(metrics_to_measure):
    if key not in my_metrics_pruned.keys():
        print(f"  {i}:\t{key}")
print()
        
print(f"Number of available metrics: {len(my_metrics_pruned)}")
print(f"Available metrics data:")
for i, key in enumerate(metrics_to_measure):
    if key in my_metrics_pruned.keys():
        print(f"  {i}:\t{key}: {my_metrics_pruned[key]}")

Number of missing metrics: 1
Missing metrics:
  8:	sm__pipe_lsu_cycles_active.avg

Number of available metrics: 20
Available metrics data:
  0:	sm__pipe_tensor_cycles_active.avg: 35029.333333333336
  1:	smsp__inst_executed_pipe_tensor.sum: 131072.0
  2:	sm__pipe_fma_cycles_active.avg: 25467.6
  3:	smsp__inst_executed_pipe_fma.sum: 382014.0
  4:	smsp__inst_executed_pipe_xu.sum: 68663.0
  5:	smsp__inst_executed_pipe_xu.avg.pct_of_peak_sustained_elapsed: 3.707795696747316
  6:	sm__pipe_alu_cycles_active.avg: 32284.6
  7:	smsp__inst_executed_pipe_alu.sum: 484269.0
  9:	smsp__inst_executed_pipe_lsu.sum: 132873.0
  10:	dram__cycles_active.avg: 9601.333333333334
  11:	lts__cycles_active.avg: 15486.833333333334
  12:	sm__inst_executed.sum: 1405579.0
  13:	sm__cycles_elapsed.avg: 123457.0
  14:	gpu__time_duration.sum: 92480.0
  15:	smsp__warps_issue_stalled_short_scoreboard.avg: 5886.8
  16:	smsp__warps_issue_stalled_long_scoreboard.avg: 7822.125
  17:	smsp__warps_issue_stalled_wait.avg: 19981.